# GPT-2 Learning Notebook

## Introduction

Welcome to this hands-on exploration of GPT-2, one of the landmark models in natural language processing. This notebook provides a practical introduction to working with GPT-2, from understanding its architecture to fine-tuning it on custom data.

**What is GPT-2?**

GPT-2 (Generative Pre-trained Transformer 2) is an autoregressive language model developed by OpenAI in 2019. It uses the Transformer architecture with a decoder-only design, trained to predict the next token in a sequence. With variants ranging from 124M to 1.5B parameters, GPT-2 demonstrated impressive zero-shot learning capabilities across various NLP tasks.

**What You'll Learn:**

- Loading and using pre-trained GPT-2 models
- Tokenization and dataset preparation
- Fine-tuning GPT-2 on custom datasets
- Understanding training parameters and their impact
- Generating text with fine-tuned models

**Prerequisites:**
- Basic understanding of deep learning concepts
- Familiarity with Python and PyTorch
- Knowledge of transformers (helpful but not required)

Let's dive in and start exploring GPT-2!

In [1]:
# Install libraries if needed
!pip install transformers datasets accelerate -q

from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

# 1. Load GPT-2 small (124M parameters)
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# IMPORTANT: GPT-2 doesn't have a padding token by default
# We need to set it to the EOS token for batch processing
tokenizer.pad_token = tokenizer.eos_token

# 2. Load a dataset (here: wikitext-2, small and easy)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Filter out empty texts to avoid issues
dataset = dataset.filter(lambda example: len(example["text"]) > 0 and not example["text"].isspace())

# 3. Tokenize the dataset with smaller sequences
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)  # Reduced from 128

tokenized_datasets = dataset.map(tokenize, batched=True, remove_columns=["text"])

# 4. Create a data collator for language modeling
# This will handle padding and creating labels automatically
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # False for causal language modeling (GPT-2 style)
)

# 5. Define training arguments with memory-optimized settings
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=1,  # Reduced to 1 to save memory
    gradient_accumulation_steps=4,  # Simulate larger batch size
    save_steps=1000,
    save_total_limit=1,  # Keep only 1 checkpoint to save disk space
    logging_steps=100,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    max_steps=500,  # Limit training steps for demo
)

# 6. Setup Trainer with smaller subset for demo
# Using a small subset to prevent memory issues
train_subset = tokenized_datasets["train"].select(range(min(1000, len(tokenized_datasets["train"]))))
eval_subset = tokenized_datasets["validation"].select(range(min(200, len(tokenized_datasets["validation"]))))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    data_collator=data_collator
)

# 7. Train
print("Starting training...")
trainer.train()

# Save the fine-tuned model
trainer.save_model("./gpt2-finetuned")
print("Training complete! Model saved to ./gpt2-finetuned")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 2461/2461 [00:02<00:00, 1010.81 examples/s]


Starting training...


/home/codespace/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


: 